# 18 - Production RAG Pipeline

---

In the previous notebooks, we learned:

- Embeddings
- Distance Metrics
- KNN
- Flat Index
- ANN
- Clustering
- IVF
- Curse of Dimensionality
- KDTree
- LSH
- Product Quantization
- IVFPQ
- HNSW
- FAISS
- Qdrant
- Hybrid Search
- Retrieval Evaluation

Now we connect everything into one complete Retrieval-Augmented Generation (RAG) pipeline.

## Think Like a Researcher

Suppose you want to build an AI assistant for company documents.

A user asks:

"What was NVIDIA's Q1 2024 revenue?"

How does the system answer this question?

Let's follow the journey step by step.

## End-to-End RAG Pipeline

```
PDFs / Documents
        │
        ▼
Document Loader
        │
        ▼
Text Chunking
        │
        ▼
Embedding Model
        │
        ▼
Vector Database
        │
        ▼
User Query
        │
        ▼
Query Embedding
        │
        ▼
Retriever
        │
        ▼
Top-K Chunks
        │
        ▼
Prompt Builder
        │
        ▼
LLM
        │
        ▼
Final Answer
```

In [1]:
documents = [
    "NVIDIA reported record revenue in Q1 2024.",
    "Apple released a new iPhone.",
    "GPU demand continues to grow."
]

## Step 1 - Chunking

Large documents are divided into smaller pieces.

Example

100-page PDF

↓

500 chunks

Why?

Embedding models have context limits,

and smaller chunks improve retrieval accuracy.

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = model.encode(documents)

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(3, 384)

## Step 2 - Store Embeddings

The embeddings are stored inside a vector database.

Examples

- Qdrant
- Pinecone
- Milvus
- Weaviate

Each embedding is stored together with metadata.

In [3]:
query = "What was NVIDIA's Q1 revenue?"

query_embedding = model.encode([query])

## Step 3 - Retrieval

The retriever compares

Query Embedding

↓

Document Embeddings

↓

Top K Chunks

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(
    query_embedding,
    embeddings
)

scores

array([[0.84347737, 0.15843494, 0.39123476]], dtype=float32)

In [5]:
import numpy as np

top_k = np.argsort(scores[0])[::-1][:2]

top_k

array([0, 2])

In [6]:
context = ""

for i in top_k:

    context += documents[i] + "\n"

print(context)

NVIDIA reported record revenue in Q1 2024.
GPU demand continues to grow.



## Step 4 - Prompt Construction

The retrieved context is added to the prompt.

Example

```
Context:

NVIDIA reported record revenue...

Question:

What was NVIDIA's Q1 revenue?

Answer only using the context.
```

## Step 5 - Generation

The LLM receives

- Context
- User Question

and generates the final answer.

Without retrieval,

the LLM relies only on its pretraining.

With retrieval,

it can answer using the latest documents.

## Complete Production Architecture

```
User

↓

Query

↓

Embedding Model

↓

Vector Database

↓

Metadata Filter

↓

Hybrid Search

↓

Top K Chunks

↓

Prompt Builder

↓

LLM

↓

Answer
```

## Production AI Stack

```
PDF

↓

Chunking

↓

Embedding Model

↓

Vector DB

↓

Retriever

↓

Re-ranker (Optional)

↓

LLM

↓

Evaluation

↓

Monitoring
```

## Popular Tools

Chunking

- LangChain
- LlamaIndex

Embeddings

- Sentence Transformers
- OpenAI
- BGE

Vector DB

- Qdrant
- Pinecone
- Milvus

Retriever

- FAISS
- Qdrant

LLM

- GPT
- Llama
- Gemini
- Claude